# 🌸 Seha Personal AI — Kaggle Launcher
Run cells from top to bottom. Add API keys, owner number, and GitHub token only in the configuration cell below. User profiles/settings are stored on GitHub only, never on Kaggle disk. Do not publish this notebook after entering credentials.

In [ ]:
# 1) CONFIGURATION — enter project values and credentials only in this Kaggle cell
# Sri Lanka examples: 0771234567, +94 77 123 4567, +94 077 123 4567, 94771234567.
OWNER_NUMBER = ""       # Owner number (used to authenticate .owner commands)
WHATSAPP_NUMBER = ""    # Exact WhatsApp account to pair; blank = OWNER_NUMBER
DEFAULT_COUNTRY_CODE = "94"
OWNER_NAME = "Owner"
GEMINI_API_KEY = ""     # Primary AI key
GROQ_API_KEY = ""       # Optional fallback key
GITHUB_OWNER = "praveenPro2012"  # Profile/settings database owner (keep your current working database)
GITHUB_REPO = "Seha-Personal-AI"    # Profile/settings database repository
GITHUB_TOKEN = ""       # REQUIRED: token with Contents Read/Write access to the database repo above
MAX_MEDIA_MB = 15

assert OWNER_NUMBER.strip(), "Enter OWNER_NUMBER first."
if not WHATSAPP_NUMBER.strip():
    WHATSAPP_NUMBER = OWNER_NUMBER
assert GEMINI_API_KEY or GROQ_API_KEY, "Enter at least one AI API key."
assert GITHUB_OWNER.strip() and GITHUB_REPO.strip() and GITHUB_TOKEN.strip(), "GitHub-only storage requires owner, repo, and token."
print(f"Configuration loaded. Source code: Sachintha919/Seha-Personal-AI. Profile/settings database: {GITHUB_OWNER}/{GITHUB_REPO}.")

In [ ]:
# 2) Install Node.js 22 and FFmpeg
import subprocess, os, sys
subprocess.run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash -", shell=True, check=True)
subprocess.run("apt-get update -qq && apt-get install -y nodejs ffmpeg git -qq", shell=True, check=True)
print(subprocess.check_output(['node','--version'], text=True).strip())
print(subprocess.check_output(['ffmpeg','-version'], text=True).splitlines()[0])

In [ ]:
# 3) Clone the updated source repository (keep database settings separate)
from pathlib import Path
import os
SOURCE_GITHUB_OWNER = "Sachintha919"
SOURCE_GITHUB_REPO = "Seha-Personal-AI"
PROJECT_DIR = Path("/kaggle/working/Seha-Personal-AI-v1.2.2")
clone_url = f"https://github.com/{SOURCE_GITHUB_OWNER}/{SOURCE_GITHUB_REPO}.git"
if (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", clone_url, str(PROJECT_DIR)], check=True)
print(f"Source code ready from {SOURCE_GITHUB_OWNER}/{SOURCE_GITHUB_REPO}.")
print(f"Profile/settings will still be stored in {GITHUB_OWNER}/{GITHUB_REPO}.")
print("The previous Seha-Personal-AI folder and its local JSON copies are left untouched.")


In [ ]:
# 4) Build the patched Baileys with its compatible libsignal, then install Seha
import json, shutil
BAILEYS_DIR = Path("/kaggle/working/Seha-Baileys")
BAILEYS_COMMIT = "c82d22d22ee01db739b781e1df064f019bff94e5"
LIBSIGNAL_COMMIT = "4d08331a833727c338c1a90041d17b870210dfae"

# Get the exact Baileys source that contains the LID/PN decrypt retry.
if BAILEYS_DIR.exists():
    shutil.rmtree(BAILEYS_DIR)
subprocess.run(
    ["git", "clone", "https://github.com/daniel-n-silva/Baileys.git", str(BAILEYS_DIR)],
    check=True
)
subprocess.run(
    ["git", "-C", str(BAILEYS_DIR), "checkout", "--detach", BAILEYS_COMMIT],
    check=True
)

# The WhatsApp stanza sometimes omits sender_pn/sender_lid on own-device
# peer sync messages. Use our own PN only when the failed LID is exactly
# the linked account's own LID.
decrypt_source_path = BAILEYS_DIR / "src" / "Utils" / "decode-wa-message.ts"
decrypt_source = decrypt_source_path.read_text()
import re

retry_pattern = re.compile(
    r"(?m)^([ \t]*)const altUser = isLidUser\(user\)\n"
    r"([ \t]+)\? stanza\.attrs\.participant_pn \|\| stanza\.attrs\.sender_pn\n"
    r"([ \t]+): stanza\.attrs\.participant_lid \|\| stanza\.attrs\.sender_lid"
)
retry_match = retry_pattern.search(decrypt_source)
if not retry_match:
    raise RuntimeError("Could not find Baileys LID retry block; refusing to build an unpatched dependency.")
base_indent, continuation_indent = retry_match.group(1), retry_match.group(2)
nested_indent = continuation_indent + continuation_indent[:1]
new_retry = "\n".join([
    base_indent + "const stanzaAltUser = isLidUser(user)",
    continuation_indent + "? stanza.attrs.participant_pn || stanza.attrs.sender_pn",
    continuation_indent + ": stanza.attrs.participant_lid || stanza.attrs.sender_lid",
    base_indent + "const ownAccountPn =",
    continuation_indent + "isLidUser(user) &&",
    continuation_indent + "isLidUser(meLid) &&",
    continuation_indent + "isJidUser(meId) &&",
    continuation_indent + "areJidsSameUser(user, meLid)",
    nested_indent + "? meId",
    nested_indent + ": undefined",
    base_indent + "const altUser = stanzaAltUser || ownAccountPn"
])
decrypt_source = decrypt_source[:retry_match.start()] + new_retry + decrypt_source[retry_match.end():]

decrypt_source_path.write_text(decrypt_source)

# Pin libsignal to the compatible commit recorded in Baileys' own yarn.lock.
baileys_package_path = BAILEYS_DIR / "package.json"
baileys_package = json.loads(baileys_package_path.read_text())
baileys_package["name"] = "@whiskeysockets/baileys"
baileys_package.setdefault("dependencies", {})["libsignal"] = (
    "git+https://github.com/whiskeysockets/libsignal-node#" + LIBSIGNAL_COMMIT
)
baileys_package_path.write_text(json.dumps(baileys_package, indent=2) + "\n")

# npm now builds Baileys using the matching libsignal API.
subprocess.run(["npm", "install"], cwd=str(BAILEYS_DIR), check=True)
BAILEYS_ENTRY = BAILEYS_DIR / "lib" / "index.js"
assert BAILEYS_ENTRY.exists(), "Baileys build did not create lib/index.js."

# It is already built. Remove prepare only from this Kaggle working copy so
# installing the local package does not trigger a second source build.
baileys_package = json.loads(baileys_package_path.read_text())
baileys_package.get("scripts", {}).pop("prepare", None)
baileys_package_path.write_text(json.dumps(baileys_package, indent=2) + "\n")

# Use the locally built package only during npm install. Restore the
# repository package.json afterwards so the next git pull stays clean.
app_package_path = PROJECT_DIR / "package.json"
original_package_json = app_package_path.read_text()
app_package = json.loads(original_package_json)
app_package["dependencies"]["@whiskeysockets/baileys"] = "file:../Seha-Baileys"
try:
    app_package_path.write_text(json.dumps(app_package, indent=2) + "\n")
    subprocess.run(["npm", "install", "--no-package-lock"], cwd=str(PROJECT_DIR), check=True)
finally:
    app_package_path.write_text(original_package_json)

subprocess.run(["npm", "run", "check"], cwd=str(PROJECT_DIR), check=True)
print(subprocess.check_output(
    ["npm", "ls", "@whiskeysockets/baileys", "--depth=0"],
    cwd=str(PROJECT_DIR), text=True
))
subprocess.run(["npm", "test"], cwd=str(PROJECT_DIR), check=True)
print("Patched Baileys installed; Seha syntax checks and tests passed.")


In [ ]:
# 5) Start Seha — pairing code appears in this cell output
assert (PROJECT_DIR / "node_modules" / "@whiskeysockets" / "baileys" / "lib" / "index.js").exists(), "Cell 4 did not finish successfully. Run cell 4 first and fix any reported error."
env = os.environ.copy()
env.update({
    "BOT_OWNER_NUMBER": OWNER_NUMBER,
    "WHATSAPP_NUMBER": WHATSAPP_NUMBER,
    "OWNER_NAME": OWNER_NAME,
    "DEFAULT_COUNTRY_CODE": DEFAULT_COUNTRY_CODE,
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "GROQ_API_KEY": GROQ_API_KEY,
    "GITHUB_OWNER": GITHUB_OWNER,
    "GITHUB_REPO": GITHUB_REPO,
    "GITHUB_TOKEN": GITHUB_TOKEN,
    "AUTH_DIR": "/kaggle/working/seha-auth",
    "MAX_MEDIA_MB": str(MAX_MEDIA_MB)
})
subprocess.run(["node", "src/main.js"], cwd=str(PROJECT_DIR), env=env, check=False)

## Pair your WhatsApp account
When the 8-digit code appears, use the **same WhatsApp account whose number you entered in `WHATSAPP_NUMBER`**.
**WhatsApp → Settings → Linked Devices → Link a Device → Link with phone number instead** → enter the code.

Sri Lankan examples accepted: `0771234567`, `+94 77 123 4567`, `+94 077 123 4567`, `94771234567`. The number must already be registered with WhatsApp.

The source code is cloned from `Sachintha919/Seha-Personal-AI`. The `GITHUB_OWNER` and `GITHUB_REPO` values in the configuration cell are used only for the profile/settings database (the current setup keeps this at `praveenPro2012/Seha-Personal-AI`). The old Kaggle project folder and its local JSON copies are not deleted. The bot does not use local copies for runtime storage; it reads and writes profile/settings records through the database repository's GitHub API. If GitHub is unavailable or the token lacks permission, saving fails rather than using local storage.

API keys, owner number, and GitHub token are supplied only in this Kaggle cell. WhatsApp auth files stay in `/kaggle/working/seha-auth` and are not uploaded to GitHub; if Kaggle resets that folder, pair again.